In [2]:
import pandas as pd 

In [4]:
data = pd.read_csv('cleaned_housingdata.csv')

In [5]:
data1 = data.copy()

In [6]:
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from langchain_groq import ChatGroq
import chromadb

C:\Users\user\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
name_collection = "faqs"
ef = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
chroma_client = chromadb.Client()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5825.27it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
data1["text"] = data1.apply(
            lambda row: f"{int(row['new_size'])} rooms house in {row['location']} with {int(row['bath'])} bathrooms and {row['total_sqft']} sqft",
            axis=1
        )

In [9]:
docs = data1['text'].to_list()

In [10]:
ids = [f"house_{i}" for i in range(len(docs))]

In [11]:
data2 = data.copy()

In [14]:
data2['text'] = data2.apply(
            lambda row: f"{int(row['new_size'])} rooms house in {row['location']} with {int(row['bath'])} bathrooms and {row['total_sqft']} sqft in price of {float(row['price'])}",
            axis=1
        )

In [29]:
metadata = [{"answer": ans} for ans in data2['text'].to_list()]
collection1 = chroma_client.get_or_create_collection(
            name=name_collection,
            embedding_function=ef
        )
batch_size = 5000

for i in range(0, len(ids), batch_size):
    collection1.add(
        ids=ids[i:i+batch_size],
        documents=docs[i:i+batch_size],
        metadatas=metadata[i:i+batch_size]
    )

In [30]:
query = "hoouses in basavangudi"

In [31]:
def get_relevent_query(query):

    collection = chroma_client.get_collection(name=name_collection)

    result = collection.query(query_texts=[query],
                              n_results=5)
    return result
query1 = "show me houseses of price range 450 to 455 in 1st block jayanagar"

result1 = get_relevent_query(query)

In [33]:
def get_embaddings(query):

    context = " ".join(r["answer"] for r in query["metadatas"][0])

    return context

context = get_embaddings(get_relevent_query(query))

print(context)

2 rooms house in Basavangudi with 2 bathrooms and 1560.0 sqft in price of 145.0 2 rooms house in Basavangudi with 2 bathrooms and 1180.0 sqft in price of 124.0 3 rooms house in Basavangudi with 3 bathrooms and 1800.0 sqft in price of 195.0 4 rooms house in Basavangudi with 3 bathrooms and 2453.0 sqft in price of 250.0 3 rooms house in Hormavu with 3 bathrooms and 1725.0 sqft in price of 85.0


In [34]:
from dotenv import load_dotenv
load_dotenv(".env.txt")

True

In [40]:
import re

def get_final_result(query, context):

    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0.2, 
        max_tokens=1000
    )

    prompt = f"""
You are a strict real estate assistant.

Your task:
- ONLY use the given context.
- DO NOT generate or assume any extra details.
- DO NOT add features, contact info, or descriptions that are not explicitly present in the context.
- If something is missing in context, leave it out.

Context:
{context}

User Query:
{query}

Instructions:
- Filter results based on the query (especially price and location).
- Extract ONLY these fields:
    - location
    - price
    - area_sqft
    - rooms
    - bathrooms
- Do NOT add anything extra.

- If no exact match:
    politely say so and show closest matches.

Output Format:
{{
  "message": "<short friendly sentence>",
  "data": [
    {{
      "location": "...",
      "price": ...,
      "area_sqft": ...,
      "rooms": ...,
      "bathrooms": ...
    }}
  ]
}}

IMPORTANT:
- Strictly return valid JSON
- No extra text
"""

    response = llm.invoke(prompt)

    return re.sub(r'\s+', ' ', response.content)

In [41]:
final_result = get_final_result(query, context)
final_result

'{ "message": "Here are the houses in Basavangudi:", "data": [ { "location": "Basavangudi", "price": 145.0, "area_sqft": 1560.0, "rooms": 2, "bathrooms": 2 }, { "location": "Basavangudi", "price": 124.0, "area_sqft": 1180.0, "rooms": 2, "bathrooms": 2 }, { "location": "Basavangudi", "price": 195.0, "area_sqft": 1800.0, "rooms": 3, "bathrooms": 3 }, { "location": "Basavangudi", "price": 250.0, "area_sqft": 2453.0, "rooms": 4, "bathrooms": 3 } ] }'